In [66]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [67]:
annual_value_adjusted = pd.read_csv("data/outputs/annual_ca_port",
                                    index_col =0, parse_dates = [1])
quarterly_value_adjusted = pd.read_csv('data/outputs/quarter_ca_port',
                                    index_col =0, parse_dates = [1])
band_value_adjusted = pd.read_csv("data/outputs/quarter_band_ca_port",
                                    index_col =0, parse_dates = [1])
asset_data = pd.read_csv("data/processed/daily_returns.csv",
                         parse_dates = [0])


In [68]:
annual_weights = pd.read_csv("data/outputs/annually_rebalanced_weights", 
                             index_col = 0, parse_dates = [1])
quarterly_weights = pd.read_csv("data/outputs/quarterly_rebalanced_weights", 
                                index_col = 0, parse_dates = [1])
band_weights = pd.read_csv("data/outputs/quarterly_rebalanced_byband_weights", 
                           index_col = 0, parse_dates = [1])

In [69]:
def portfolio_value_deconstruction(weights_df, portfolio_df):

    starting_weights = weights_df[weights_df['weight_type'] == 'starting']

    start_dates = weights_df.loc[weights_df['weight_type'] == 'starting', 'Date']

    end_dates = weights_df.loc[weights_df['weight_type'] == 'ending', 'Date']

    returns_list = []

    for row in range(len(start_dates)):

        start = start_dates.iloc[row]
        
        end = end_dates.iloc[row]

        period_data = portfolio_df[portfolio_df['Date'].between(start, end)].reset_index()

        weights = starting_weights[starting_weights['Date'] == start].drop(columns = ['Date','weight_type'])

        for row in range(len(period_data)):

            returns_by_asset = weights.iloc[0] * period_data['portfolio_return'][row]

            returns_by_asset['Date'] = period_data['Date'].iloc[row]

            returns_list.append(returns_by_asset)

    return pd.DataFrame(returns_list).set_index('Date')


In [70]:

annual_portfolio_asset_contributions = portfolio_value_deconstruction(annual_weights, annual_value_adjusted)
quarterly_portfolio_asset_contributions = portfolio_value_deconstruction(quarterly_weights, quarterly_value_adjusted)
banded_portfolio_asset_contributions = portfolio_value_deconstruction(band_weights, band_value_adjusted)

In [72]:
def asset_contributions_to_returns(portfolio_df, asset_contribution_df):
    weights_sums = []
    total_return = portfolio_df['portfolio_return'].sum()

    for col in asset_contribution_df.columns:
        name = asset_contribution_df[col].name
        col_sum = asset_contribution_df[col].sum()
        percent_cont = col_sum / total_return
        weights_sums.append({
            'asset' : name, 
            'total_return' : col_sum,
            'percent_contribution' : (f'{percent_cont * 100 :.3f} %')
        })

    return pd.DataFrame(weights_sums)

asset_contribution_table_band = asset_contributions_to_returns(band_value_adjusted, banded_portfolio_asset_contributions)